---Converting categorical features to numerical features using one hot encoding---

In [2]:
import pandas as pd

# Load feature-engineered dataset
df = pd.read_csv("college_iot_energy_monitoring.csv")

# Identify categorical columns
categorical_cols = df.select_dtypes(
    include=["object", "category"]
).columns.tolist()

print("Categorical columns:", categorical_cols)

# Apply One-Hot Encoding
df_encoded = pd.get_dummies(
    df,
    columns=categorical_cols,
    dtype=int
)

# Display results
print(df_encoded.head())
print("New shape:", df_encoded.shape)

# Save encoded dataset
df_encoded.to_csv(
    "college_iot_energy_monitoring_onehot_encoded.csv",
    index=False
)

Categorical columns: ['device']
   hour  day  temperature  humidity         power  duration    energy  \
0     9   29        22.27     39.65   2115.406021    4.1982  0.104992   
1    21   20        30.03     40.42    569.237511    5.4644  0.134754   
2     6    3        29.24     61.64   2990.349781    1.3133  0.028226   
3     8   19        22.10     95.00  28833.701030    2.1353  0.116440   
4    16   22        21.82     38.87   2051.381944    1.4630  0.025927   

   device_Agriculture Block - Climate Control Load  \
0                                                0   
1                                                0   
2                                                0   
3                                                0   
4                                                0   

   device_Agriculture Block - Crop Monitoring Load  \
0                                                0   
1                                                1   
2                                         

---Linear Regression---

In [3]:

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# 1. Load encoded dataset
df = pd.read_csv("college_iot_energy_monitoring_onehot_encoded.csv")

# 2. Separate features and target
X = df.drop(columns=["energy"])
y = df["energy"]

# 3. Split into training and testing data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 4. Train basic Linear Regression model
model = LinearRegression()
model.fit(X_train, y_train)

# 5. Predict energy consumption
y_pred = model.predict(X_test)

# 6. Evaluate the model
print("MAE:", mean_absolute_error(y_test, y_pred))
print("MSE:", mean_squared_error(y_test, y_pred))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred)))
print("R2 Score:", r2_score(y_test, y_pred))

# 7. Compare actual and predicted values
results = pd.DataFrame({
    "Actual Energy": y_test.values,
    "Predicted Energy": y_pred
})

print(results.head(10))


MAE: 0.04043521673344037
MSE: 0.002506843531712114
RMSE: 0.05006838854718727
R2 Score: 0.47090243802450715
   Actual Energy  Predicted Energy
0       0.250056          0.201711
1       0.246698          0.210335
2       0.010479          0.097674
3       0.005000          0.084229
4       0.009773          0.084764
5       0.196259          0.143102
6       0.092012          0.119517
7       0.092392          0.103991
8       0.096023          0.062982
9       0.111247          0.078485


---Standardizing the dataset---

In [4]:

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# 1. Load one-hot-encoded data
df = pd.read_csv("college_iot_energy_monitoring_onehot_encoded.csv")

# 2. Separate features and target
X = df.drop(columns=["energy"])
y = df["energy"]

# 3. Split BEFORE standardization to prevent data leakage
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 4. Standardize input features
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 5. Train Linear Regression
model = LinearRegression()
model.fit(X_train_scaled, y_train)

# 6. Predict
y_pred = model.predict(X_test_scaled)

# 7. Evaluate
print("Linear Regression with Standardization")
print("--------------------------------------")
print("MAE:", mean_absolute_error(y_test, y_pred))
print("MSE:", mean_squared_error(y_test, y_pred))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred)))
print("R2 Score:", r2_score(y_test, y_pred))

# 8. Compare actual and predicted values
results = pd.DataFrame({
    "Actual Energy": y_test.values,
    "Predicted Energy": y_pred
})
print(results.head(10))


Linear Regression with Standardization
--------------------------------------
MAE: 0.040435216733440374
MSE: 0.002506843531712115
RMSE: 0.05006838854718729
R2 Score: 0.47090243802450704
   Actual Energy  Predicted Energy
0       0.250056          0.201711
1       0.246698          0.210335
2       0.010479          0.097674
3       0.005000          0.084229
4       0.009773          0.084764
5       0.196259          0.143102
6       0.092012          0.119517
7       0.092392          0.103991
8       0.096023          0.062982
9       0.111247          0.078485


---applying Regularization---

In [16]:

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV, KFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# 1. Load dataset
df = pd.read_csv("college_iot_energy_monitoring_onehot_encoded.csv")

# 2. Separate features and target
X = df.drop(columns=["energy"])
y = df["energy"]

# 3. Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 4. Five-fold cross-validation
cv = KFold(n_splits=5, shuffle=True, random_state=42)

# 5. Baseline: ordinary Linear Regression
baseline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LinearRegression())
])
baseline.fit(X_train, y_train)
baseline_pred = baseline.predict(X_test)

# 6. Define regularized models and alpha values
models = {
    "Ridge (L2)": (
        Ridge(),
        {"model__alpha": [0.001, 0.01, 0.1, 1, 10, 100]}
    ),
    "Lasso (L1)": (
        Lasso(max_iter=50000),
        {"model__alpha": [0.00001, 0.0001, 0.001, 0.01, 0.1, 1]}
    ),
    "Elastic Net": (
        ElasticNet(max_iter=50000),
        {
            "model__alpha": [0.0001, 0.001, 0.01, 0.1, 1],
            "model__l1_ratio": [0.2, 0.5, 0.8]
        }
    )
}

results = []

def evaluate(name, y_true, y_pred, best_params=None, cv_rmse=None):
    mse = mean_squared_error(y_true, y_pred)
    results.append({
        "Model": name,
        "Best Parameters": best_params,
        "CV RMSE": cv_rmse,
        "Test MAE": mean_absolute_error(y_true, y_pred),
        "Test RMSE": np.sqrt(mse),
        "Test R2": r2_score(y_true, y_pred)
    })

evaluate("Ordinary Linear Regression", y_test, baseline_pred)

# 7. Tune each regularized model
best_models = {}

for name, (estimator, param_grid) in models.items():
    pipeline = Pipeline([
        ("scaler", StandardScaler()),
        ("model", estimator)
    ])

    search = GridSearchCV(
        estimator=pipeline,
        param_grid=param_grid,
        scoring="neg_mean_squared_error",
        cv=cv,
        n_jobs=-1
    )

    search.fit(X_train, y_train)

    # Evaluate the selected model on the held-out test set
    pred = search.predict(X_test)
    best_models[name] = search.best_estimator_

    evaluate(
        name,
        y_test,
        pred,
        best_params=search.best_params_,
        cv_rmse=np.sqrt(-search.best_score_)
    )

    print(f"\n{name}")
    print("Best parameters:", search.best_params_)
    print("Best CV RMSE:", np.sqrt(-search.best_score_))

# 8. Compare all models
comparison = pd.DataFrame(results)
print("\nFinal Comparison")
print(comparison.to_string(index=False))



Ridge (L2)
Best parameters: {'model__alpha': 100}
Best CV RMSE: 0.06343084570019593

Lasso (L1)
Best parameters: {'model__alpha': 0.001}
Best CV RMSE: 0.06333997394481984

Elastic Net
Best parameters: {'model__alpha': 0.001, 'model__l1_ratio': 0.8}
Best CV RMSE: 0.06334244012548995

Final Comparison
                     Model                                 Best Parameters  CV RMSE  Test MAE  Test RMSE  Test R2
Ordinary Linear Regression                                            None      NaN  0.040435   0.050068 0.470902
                Ridge (L2)                           {'model__alpha': 100} 0.063431  0.040841   0.050435 0.463135
                Lasso (L1)                         {'model__alpha': 0.001} 0.063340  0.040936   0.050471 0.462358
               Elastic Net {'model__alpha': 0.001, 'model__l1_ratio': 0.8} 0.063342  0.040834   0.050379 0.464328


---Using Polynomial Regression---

In [5]:

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# 1. Load one-hot-encoded dataset
df = pd.read_csv("college_iot_energy_monitoring_onehot_encoded.csv")

# 2. Separate features and target
X = df.drop(columns=["energy"])
y = df["energy"]

# 3. Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 4. Polynomial Regression model (degree 2)
model = Pipeline([
    ("scaler_before_poly", StandardScaler()),
    ("polynomial", PolynomialFeatures(degree=2, include_bias=False)),
    ("scaler_after_poly", StandardScaler()),
    ("regression", LinearRegression())
])

# 5. Train
model.fit(X_train, y_train)

# 6. Predict
y_pred = model.predict(X_test)

# 7. Evaluate
print("Polynomial Regression (Degree 2)")
print("--------------------------------")
print("MAE:", mean_absolute_error(y_test, y_pred))
print("MSE:", mean_squared_error(y_test, y_pred))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred)))
print("R2 Score:", r2_score(y_test, y_pred))

# 8. Compare predictions
results = pd.DataFrame({
    "Actual Energy": y_test.values,
    "Predicted Energy": y_pred
})
print(results.head(10))


Polynomial Regression (Degree 2)
--------------------------------
MAE: 0.0408562538897185
MSE: 0.0025246673589024095
RMSE: 0.050246068093955466
R2 Score: 0.4671405185460239
   Actual Energy  Predicted Energy
0       0.250056          0.250888
1       0.246698          0.236250
2       0.010479          0.126304
3       0.005000          0.075473
4       0.009773          0.108113
5       0.196259          0.109770
6       0.092012          0.113543
7       0.092392          0.095868
8       0.096023          0.061716
9       0.111247          0.077506


---KNN Algorithm---

In [13]:

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# 1. Load one-hot-encoded dataset
df = pd.read_csv("college_iot_energy_monitoring_onehot_encoded.csv")

# 2. Separate features and target
X = df.drop(columns=["energy"])
y = df["energy"]

# 3. Split the dataset
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 4. Build KNN Regression pipeline
model = Pipeline([
    ("scaler", StandardScaler()),
    ("knn", KNeighborsRegressor(n_neighbors=17, weights="uniform"))
])

# 5. Train the model
model.fit(X_train, y_train)

# 6. Predict energy consumption
y_pred = model.predict(X_test)

# 7. Evaluate
print("KNN Regression Results")
print("-----------------------")
print("MAE:", mean_absolute_error(y_test, y_pred))
print("MSE:", mean_squared_error(y_test, y_pred))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred)))
print("R2 Score:", r2_score(y_test, y_pred))

# 8. Compare actual and predicted values
results = pd.DataFrame({
    "Actual Energy": y_test.values,
    "Predicted Energy": y_pred
})
print(results.head(10))


KNN Regression Results
-----------------------
MAE: 0.042679051360294123
MSE: 0.002705632268813849
RMSE: 0.05201569252460116
R2 Score: 0.4289458360993441
   Actual Energy  Predicted Energy
0       0.250056          0.190846
1       0.246698          0.233078
2       0.010479          0.113259
3       0.005000          0.076121
4       0.009773          0.101316
5       0.196259          0.097470
6       0.092012          0.097127
7       0.092392          0.134975
8       0.096023          0.100303
9       0.111247          0.071556


----SVR wih rbf----

In [14]:

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

# 1. Load dataset
df = pd.read_csv(
    "college_iot_energy_monitoring_onehot_encoded.csv"
)

# 2. Separate features and target
X = df.drop(columns=["energy"])
y = df["energy"]

# 3. Split the data
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

# 4. Create SVR model with RBF kernel
model = Pipeline([
    ("scaler", StandardScaler()),
    ("svr", SVR(
        kernel="rbf",
        C=1.0,
        epsilon=0.1,
        gamma="scale"
    ))
])

# 5. Train the model
model.fit(X_train, y_train)

# 6. Make predictions
y_pred = model.predict(X_test)

# 7. Evaluate the model
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print("SVR with RBF Kernel")
print("-------------------")
print("MAE:", mae)
print("MSE:", mse)
print("RMSE:", rmse)
print("R2 Score:", r2)

# 8. Compare actual and predicted energy
results = pd.DataFrame({
    "Actual Energy": y_test.values,
    "Predicted Energy": y_pred
})

print("\nSample Predictions:")
print(results.head(10))


SVR with RBF Kernel
-------------------
MAE: 0.04564989596906395
MSE: 0.0030074967026736356
RMSE: 0.05484064826999801
R2 Score: 0.3652339474306311

Sample Predictions:
   Actual Energy  Predicted Energy
0       0.250056          0.186986
1       0.246698          0.294570
2       0.010479          0.129283
3       0.005000          0.097027
4       0.009773          0.107541
5       0.196259          0.132926
6       0.092012          0.120571
7       0.092392          0.116937
8       0.096023          0.035060
9       0.111247          0.118559


----Hyper Parameter Tuning of both KNN and SVM---

In [15]:

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV, KFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

# 1. Load the one-hot-encoded dataset
df = pd.read_csv(
    "college_iot_energy_monitoring_onehot_encoded.csv"
)

# 2. Separate features and target
X = df.drop(columns=["energy"])
y = df["energy"]

# 3. Split before tuning to avoid test-data leakage
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

# 4. Cross-validation strategy
cv = KFold(n_splits=5, shuffle=True, random_state=42)

# 5. KNN pipeline
knn_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", KNeighborsRegressor())
])

knn_params = {
    "model__n_neighbors": [3, 5, 7, 9, 11, 15, 20],
    "model__weights": ["uniform", "distance"],
    "model__p": [1, 2]
}

knn_search = GridSearchCV(
    estimator=knn_pipeline,
    param_grid=knn_params,
    scoring="neg_mean_squared_error",
    cv=cv,
    n_jobs=-1
)

knn_search.fit(X_train, y_train)

best_knn = knn_search.best_estimator_
knn_pred = best_knn.predict(X_test)

# 6. SVM/SVR with RBF kernel
svr_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", SVR(kernel="rbf"))
])

svr_params = {
    "model__C": [0.1, 1, 10, 100],
    "model__epsilon": [0.01, 0.05, 0.1, 0.2],
    "model__gamma": ["scale", 0.01, 0.1, 1]
}

svr_search = GridSearchCV(
    estimator=svr_pipeline,
    param_grid=svr_params,
    scoring="neg_mean_squared_error",
    cv=cv,
    n_jobs=-1
)

svr_search.fit(X_train, y_train)

best_svr = svr_search.best_estimator_
svr_pred = best_svr.predict(X_test)

# 7. Evaluation function
def evaluate_model(name, y_true, y_pred, search):
    mse = mean_squared_error(y_true, y_pred)

    print(f"\n{name}")
    print("-" * 40)
    print("Best Parameters:", search.best_params_)
    print("Best CV RMSE:", np.sqrt(-search.best_score_))
    print("Test MAE:", mean_absolute_error(y_true, y_pred))
    print("Test MSE:", mse)
    print("Test RMSE:", np.sqrt(mse))
    print("Test R2:", r2_score(y_true, y_pred))

# 8. Print results
evaluate_model("Tuned KNN Regression", y_test, knn_pred, knn_search)
evaluate_model("Tuned SVR (RBF Kernel)", y_test, svr_pred, svr_search)

# 9. Compare both models
comparison = pd.DataFrame({
    "Model": ["KNN Regression", "SVR (RBF Kernel)"],
    "Best CV RMSE": [
        np.sqrt(-knn_search.best_score_),
        np.sqrt(-svr_search.best_score_)
    ],
    "Test MAE": [
        mean_absolute_error(y_test, knn_pred),
        mean_absolute_error(y_test, svr_pred)
    ],
    "Test RMSE": [
        np.sqrt(mean_squared_error(y_test, knn_pred)),
        np.sqrt(mean_squared_error(y_test, svr_pred))
    ],
    "Test R2": [
        r2_score(y_test, knn_pred),
        r2_score(y_test, svr_pred)
    ]
})

print("\nFinal Model Comparison")
print(comparison.to_string(index=False))



Tuned KNN Regression
----------------------------------------
Best Parameters: {'model__n_neighbors': 20, 'model__p': 1, 'model__weights': 'distance'}
Best CV RMSE: 0.06360995775768316
Test MAE: 0.041826489915581196
Test MSE: 0.0026228643207099767
Test RMSE: 0.051213907493082154
Test R2: 0.4464149437630218

Tuned SVR (RBF Kernel)
----------------------------------------
Best Parameters: {'model__C': 0.1, 'model__epsilon': 0.01, 'model__gamma': 0.01}
Best CV RMSE: 0.06199897954850877
Test MAE: 0.038971679765861404
Test MSE: 0.0023440811605918935
Test RMSE: 0.048415711918672576
Test R2: 0.5052552696438956

Final Model Comparison
           Model  Best CV RMSE  Test MAE  Test RMSE  Test R2
  KNN Regression      0.063610  0.041826   0.051214 0.446415
SVR (RBF Kernel)      0.061999  0.038972   0.048416 0.505255
